# Scenario: Medical Image Classification
You’re training a convolutional neural network (CNN) to detect pneumonia from chest X-rays.
- Training accuracy: 95%
- Validation accuracy: 74%
At first glance, the model seems powerful — it almost perfectly classifies the training set. But the sharp drop in validation accuracy signals overfitting: the network has memorized the training images (specific pixel patterns, noise, or even hospital-specific artifacts) instead of learning generalizable features of pneumonia.

⚙️ Levers to Address Overfitting
- Data Augmentation: Rotate, flip, and adjust brightness of X-rays to simulate variability.
- Regularization: Apply dropout in dense layers or L2 weight decay.
- Transfer Learning: Use a pretrained backbone (e.g., ResNet) to leverage generalized features.
- Cross-validation: Ensure robustness across different patient subsets.
- Early Stopping: Halt training when validation loss stops improving


 1. Data Preparation
-----------------------------
 Assume spectrograms are stored as images in folders:
 dataset_root/
   ├── train/
   │   ├── rock/
   │   ├── jazz/
   │   ├── classical/
   │   ├── hiphop/
   │   └── electronic/
   └── val/ (same structure)

In [1]:
# ============================================================
# Scenario: Fine-tune ResNet-50 for Music Genre Classification
# ============================================================

import os
import numpy as np
import tensorflow as tf
from PIL import Image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

# ============================================================
# 1. Create Random Dataset (Spectrogram Images)
# ============================================================

dataset_root = "dataset_root"
train_dir = os.path.join(dataset_root, "train")
val_dir = os.path.join(dataset_root, "val")

genres = ["rock", "jazz", "classical", "hiphop", "electronic"]

for split in [train_dir, val_dir]:
    for genre in genres:
        os.makedirs(os.path.join(split, genre), exist_ok=True)

def create_random_images(folder, num_images):
    for i in range(num_images):
        img = np.random.randint(0,255,(224,224,3),dtype=np.uint8)
        img = Image.fromarray(img)
        img.save(os.path.join(folder,f"{i}.jpg"))

for genre in genres:
    create_random_images(os.path.join(train_dir,genre),100)
    create_random_images(os.path.join(val_dir,genre),20)

# ============================================================
# 2. Data Generators (with augmentation)
# ============================================================

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

# ============================================================
# 3. Load Pretrained ResNet50
# ============================================================

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze pretrained layers
for layer in base_model.layers:
    layer.trainable = False

# ============================================================
# 4. Add Custom Classification Layers
# ============================================================

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
predictions = Dense(5, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=predictions)

# ============================================================
# 5. Compile Model
# ============================================================

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# ============================================================
# 6. Early Stopping (to reduce overfitting)
# ============================================================

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

# ============================================================
# 7. Train Model
# ============================================================

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[early_stop]
)

# ============================================================
# 8. Evaluate Model
# ============================================================

loss, accuracy = model.evaluate(val_generator)

print("Validation Accuracy:", accuracy)

Found 500 images belonging to 5 classes.
Found 100 images belonging to 5 classes.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 39s 2s/step - accuracy: 0.2104 - loss: 1.9191 - val_accuracy: 0.2000 - val_loss: 1.6853
Epoch 2/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 498ms/step - accuracy: 0.2162 - loss: 1.7529 - val_accuracy: 0.2000 - val_loss: 1.6468
Epoch 3/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 417ms/step - accuracy: 0.1893 - loss: 1.6854 - val_accuracy: 0.2000 - val_loss: 1.6292
Epoch 4/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 501ms/step - accuracy: 0.1823 - loss: 1.6660 - val_accuracy: 0.2000 - val_loss: 1.6098
Epoch 5/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 427ms/step - accuracy: 0.1770 - loss: 1.6179 - val_accuracy: 0.2000 - val_loss: 1.6095
Epoch 6/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 494ms/step - accuracy: 0.1939 - loss: 1.6193 - val_accuracy: 0.2000 - val_loss: 1.6110
Epoch 7/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 419ms/step - accuracy: 0.2258 - loss: 1.6092 - val_accuracy: 0.2000 - val_loss: 1.6104
Epoch 8/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 434ms/step - accuracy: 0.1915 - loss: 1.6094 - val_accuracy: 0.20